# 🔮 SARIMA — พยากรณ์คำขอกองทุนยุติธรรมรายเดือน

**Goal:** ใช้ time-series model พยากรณ์จำนวนคำขอล่วงหน้า 12 เดือน

**Model:** SARIMA(1,1,1)(1,1,1)₁₂

**Evaluation:**
- Train/test split: กันข้อมูล 6 เดือนสุดท้ายไว้ทดสอบ
- Metrics: MAE, RMSE, MAPE
- จากนั้น refit ข้อมูลทั้งหมดเพื่อพยากรณ์ 12 เดือนข้างหน้า

In [ ]:
# Google Colab — run this cell first
!pip install pandas plotly openpyxl statsmodels -q

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

df = pd.read_excel('66-68Stat_ServiceType.xlsx')
df['RequestAmount'] = pd.to_numeric(df['RequestAmount'], errors='coerce').fillna(0)
df['TotalAmount'] = pd.to_numeric(df['TotalAmount'], errors='coerce').fillna(0)
df['date'] = pd.to_datetime(df[['Year']].assign(month=df['Month'], day=1).rename(
    columns={'Year':'year'}))

# Monthly aggregation
monthly = df.groupby('date')['CaseAmount'].sum().sort_index()
monthly.index = pd.DatetimeIndex(monthly.index, freq='MS')
print(f'Monthly series: {len(monthly)} months')
print(f'Date range: {monthly.index.min():%Y-%m} → {monthly.index.max():%Y-%m}')
print(f'Mean: {monthly.mean():.0f} | Std: {monthly.std():.0f}')
monthly.head()

## Seasonal Decomposition

In [ ]:
# Decompose (use additive since series is relatively stable)
decomp = seasonal_decompose(monthly, model='additive', period=12)

fig_decomp = go.Figure()
for name, data, color in [
    ('Observed', decomp.observed, '#264653'),
    ('Trend', decomp.trend, '#e76f51'),
    ('Seasonal', decomp.seasonal, '#2a9d8f'),
    ('Residual', decomp.resid, '#e9c46a'),
]:
    fig_decomp.add_trace(go.Scatter(
        x=data.index, y=data.values, mode='lines',
        name=name, line=dict(color=color, width=2),
    ))

fig_decomp.update_layout(
    title='Seasonal Decomposition — คำขอรายเดือน',
    xaxis_title='เดือน', yaxis_title='คำขอ',
    template='plotly_white', height=450,
    font=dict(family='Sarabun, sans-serif'),
    legend=dict(orientation='h', y=1.12),
)
fig_decomp.show()

## Train / Test Split

In [ ]:
H = 6  # holdout last 6 months
train, test = monthly.iloc[:-H], monthly.iloc[-H:]
print(f'Train: {len(train)} months ({train.index.min():%Y-%m} → {train.index.max():%Y-%m})')
print(f'Test:  {len(test)} months ({test.index.min():%Y-%m} → {test.index.max():%Y-%m})')

## SARIMA Model — Holdout Evaluation

In [ ]:
# Fit SARIMA on train
model_ho = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12),
    enforce_stationarity=False, enforce_invertibility=False)
fit_ho = model_ho.fit(disp=False)
print(fit_ho.summary().tables[0])

# Forecast holdout period
fc_ho = fit_ho.get_forecast(H)
pred_mean = fc_ho.predicted_mean
pred_ci = fc_ho.conf_int(alpha=0.05)

# Metrics
mae = np.mean(np.abs(test.values - pred_mean.values))
rmse = np.sqrt(np.mean((test.values - pred_mean.values)**2))
mape = np.mean(np.abs((test.values - pred_mean.values) / test.values)) * 100
print(f'\n📊 Holdout Evaluation ({H} months):')
print(f'  MAE  = {mae:.0f}')
print(f'  RMSE = {rmse:.0f}')
print(f'  MAPE = {mape:.1f}%')

### Chart 1 — Actual vs Predicted (Holdout)

In [ ]:
fig_ho = go.Figure()
fig_ho.add_trace(go.Scatter(x=train.index, y=train.values,
    mode='lines', name='Train', line=dict(color='#264653', width=2)))
fig_ho.add_trace(go.Scatter(x=test.index, y=test.values,
    mode='lines+markers', name='Actual (test)', line=dict(color='#2a9d8f', width=2.5)))
fig_ho.add_trace(go.Scatter(x=pred_mean.index, y=pred_mean.values,
    mode='lines+markers', name='Predicted', line=dict(color='#e76f51', width=2.5, dash='dash')))

# Confidence interval
fig_ho.add_trace(go.Scatter(
    x=list(pred_ci.index) + list(pred_ci.index[::-1]),
    y=list(pred_ci.iloc[:,1]) + list(pred_ci.iloc[:,0][::-1]),
    fill='toself', fillcolor='rgba(231,111,81,0.15)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI',
    hoverinfo='skip'))

fig_ho.update_layout(
    title=f'Holdout: Actual vs Predicted (MAE={mae:.0f}, MAPE={mape:.1f}%)',
    xaxis_title='เดือน', yaxis_title='จำนวนคำขอ',
    template='plotly_white', height=450,
    font=dict(family='Sarabun, sans-serif'),
    legend=dict(orientation='h', y=1.12),
)
fig_ho.show()

## Full Refit → Forecast 12 เดือนข้างหน้า

In [ ]:
# Refit on full data
model_full = SARIMAX(monthly, order=(1,1,1), seasonal_order=(1,1,1,12),
    enforce_stationarity=False, enforce_invertibility=False)
fit_full = model_full.fit(disp=False)

FC = 12
fc_full = fit_full.get_forecast(FC)
fc_mean = fc_full.predicted_mean
fc_ci = fc_full.conf_int(alpha=0.05)

print(f'📅 Forecast {FC} months ahead: {fc_mean.index[0]:%Y-%m} → {fc_mean.index[-1]:%Y-%m}')
print(f'\nPredicted monthly cases:')
for dt, val in zip(fc_mean.index, fc_mean.values):
    print(f'  {dt:%Y-%m}: {val:,.0f}')

### Chart 2 — Full Forecast 12 เดือน

In [ ]:
fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=monthly.index, y=monthly.values,
    mode='lines', name='Historical', line=dict(color='#264653', width=2)))
fig_fc.add_trace(go.Scatter(x=fc_mean.index, y=fc_mean.values,
    mode='lines+markers', name='Forecast', line=dict(color='#e76f51', width=2.5)))

fig_fc.add_trace(go.Scatter(
    x=list(fc_ci.index) + list(fc_ci.index[::-1]),
    y=list(fc_ci.iloc[:,1]) + list(fc_ci.iloc[:,0][::-1]),
    fill='toself', fillcolor='rgba(231,111,81,0.15)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI',
    hoverinfo='skip'))

fig_fc.update_layout(
    title=f'SARIMA Forecast — {FC} เดือนข้างหน้า (2568+)',
    xaxis_title='เดือน', yaxis_title='จำนวนคำขอ',
    template='plotly_white', height=450,
    font=dict(family='Sarabun, sans-serif'),
    legend=dict(orientation='h', y=1.12),
)
fig_fc.show()

## สรุป

SARIMA(1,1,1)(1,1,1)₁₂ ใช้พยากรณ์จำนวนคำขอกองทุนยุติธรรมรายเดือน

- **Holdout evaluation:** ทดสอบกับข้อมูล 6 เดือนสุดท้าย ได้ MAE และ MAPE ตามที่แสดง
- **Forecast:** พยากรณ์ 12 เดือนข้างหน้าพร้อมช่วงความเชื่อมั่น 95%
- **Seasonal pattern:** มีรูปแบบตามเดือนที่ชัดเจน